In [1]:
import pandas as pd
import numpy as np

In [2]:
import yfinance as yf

In [3]:
import time
from datetime import date

In [4]:
tickers = [
    "RY.TO",    # Banking
    "TD.TO",    # Banking
    "SHOP.TO",  # Technology
    "ENB.TO",   # Energy / Pipeline
    "CNR.TO",   # Transportation
    "BMO.TO",   # Banking
    "CNQ.TO",   # Energy
    "SU.TO",    # Energy
    "BN.TO",    # Financial Services
    "ABX.TO"    # Mining
]

all_stocks = []

for ticker in tickers:

    print(f"Downloading {ticker}...")

    df = yf.download(
        ticker,
        start="2000-01-01",
        interval="1d",
        auto_adjust=False,
        actions=True,
        progress=False
    )

    if df.empty:
        print(f"No data found for {ticker}")
        continue

    # Handle MultiIndex columns if returned
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df["Ticker"] = ticker

    all_stocks.append(df)

# Combine all stocks
stocks = pd.concat(all_stocks, ignore_index=True)

# Organize columns
stocks = stocks[
    [
        "Date",
        "Ticker",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "Dividends",
        "Stock Splits"
    ]
]

stocks = stocks.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Shape:", stocks.shape)
print(stocks.head())

stocks.to_csv("canada_stocks_daily.csv", index=False)

Shape: (63575, 10)
Price       Date  Ticker       Open   High        Low      Close  Adj Close  \
0     2000-01-04  ABX.TO  25.150000  25.65  24.950001  25.450001  17.804482   
1     2000-01-05  ABX.TO  25.400000  26.00  25.350000  25.650000  17.944401   
2     2000-01-06  ABX.TO  25.799999  26.00  25.500000  25.799999  18.049345   
3     2000-01-07  ABX.TO  25.850000  26.00  25.100000  25.250000  17.664574   
4     2000-01-10  ABX.TO  25.350000  25.35  25.150000  25.250000  17.664574   

Price   Volume  Dividends  Stock Splits  
0       926500        0.0           0.0  
1      1452200        0.0           0.0  
2       600200        0.0           0.0  
3      1020800        0.0           0.0  
4      1281100        0.0           0.0  


In [5]:
print("Shape:", stocks.shape)

print("\nDate range:")
print(stocks["Date"].min(), stocks["Date"].max())

print("\nStocks:")
print(stocks["Ticker"].nunique())
print(stocks["Ticker"].unique())

print("\nMissing values:")
print(stocks.isnull().sum())

print("\nDuplicate rows:")
print(stocks.duplicated(["Date", "Ticker"]).sum())

print("\nRows per stock:")
print(stocks.groupby("Ticker").size())

print("\nDate range per stock:")
print(
    stocks.groupby("Ticker")["Date"]
    .agg(["min", "max", "count"])
)

Shape: (63575, 10)

Date range:
2000-01-04 00:00:00 2026-11-06 00:00:00

Stocks:
10
['ABX.TO' 'BMO.TO' 'BN.TO' 'CNQ.TO' 'CNR.TO' 'ENB.TO' 'RY.TO' 'SHOP.TO'
 'SU.TO' 'TD.TO']

Missing values:
Price
Date            0
Ticker          0
Open            0
High            0
Low             0
Close           0
Adj Close       0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

Duplicate rows:
0

Rows per stock:
Ticker
ABX.TO     6744
BMO.TO     6744
BN.TO      6744
CNQ.TO     6744
CNR.TO     6744
ENB.TO     6745
RY.TO      6744
SHOP.TO    2878
SU.TO      6744
TD.TO      6744
dtype: int64

Date range per stock:
               min        max  count
Ticker                              
ABX.TO  2000-01-04 2026-11-06   6744
BMO.TO  2000-01-04 2026-11-06   6744
BN.TO   2000-01-04 2026-11-06   6744
CNQ.TO  2000-01-04 2026-11-06   6744
CNR.TO  2000-01-04 2026-11-06   6744
ENB.TO  2000-01-04 2026-11-06   6745
RY.TO   2000-01-04 2026-11-06   6744
SHOP.TO 2015-05-21 2026-11-06   2878
S

In [6]:
print("Invalid prices:")
print(
    stocks[
        (stocks["Open"] <= 0) |
        (stocks["High"] <= 0) |
        (stocks["Low"] <= 0) |
        (stocks["Close"] <= 0)
    ].shape
)

Invalid prices:
(1, 10)


In [7]:
invalid_ohlc = stocks[
    (stocks["High"] < stocks["Low"]) |
    (stocks["High"] < stocks["Open"]) |
    (stocks["High"] < stocks["Close"]) |
    (stocks["Low"] > stocks["Open"]) |
    (stocks["Low"] > stocks["Close"])
]

print("Invalid OHLC rows:", len(invalid_ohlc))

Invalid OHLC rows: 0


In [8]:
print("Zero volume:", (stocks["Volume"] == 0).sum())
print("Negative volume:", (stocks["Volume"] < 0).sum())

Zero volume: 5
Negative volume: 0


In [9]:
invalid_prices = stocks[
    (stocks["Open"] <= 0) |
    (stocks["High"] <= 0) |
    (stocks["Low"] <= 0) |
    (stocks["Close"] <= 0)
]

zero_volume = stocks[stocks["Volume"] == 0]

print("Invalid price records:")
display(invalid_prices)

print("Zero volume records:")
display(zero_volume)

Invalid price records:


Price,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
34570,2003-05-19,ENB.TO,0.0,0.0,0.0,0.0,0.0,0,0.10375,0.0


Zero volume records:


Price,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
20658,2001-09-11,CNQ.TO,2.812194,2.812194,2.812194,2.812194,1.458967,0,0.00000,0.0
31643,2018-07-30,CNR.TO,117.779999,117.779999,117.779999,117.779999,100.075592,0,0.00000,0.0
34146,2001-09-11,ENB.TO,10.325000,10.325000,10.325000,10.325000,3.414146,0,0.00000,0.0
34570,2003-05-19,ENB.TO,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.10375,0.0
50513,2001-09-11,SU.TO,11.230000,11.230000,11.230000,11.230000,6.324641,0,0.00000,0.0


In [10]:
stocks = stocks[
    ~(
        (stocks["Open"] <= 0) |
        (stocks["High"] <= 0) |
        (stocks["Low"] <= 0) |
        (stocks["Close"] <= 0)
    )
].copy()

In [11]:
print("Shape:", stocks.shape)

print(
    "Invalid prices:",
    (
        (stocks["Open"] <= 0) |
        (stocks["High"] <= 0) |
        (stocks["Low"] <= 0) |
        (stocks["Close"] <= 0)
    ).sum()
)

print("Zero volume:", (stocks["Volume"] == 0).sum())

Shape: (63574, 10)
Invalid prices: 0
Zero volume: 4


In [12]:
stock_summary = stocks.groupby("Ticker").agg(
    Records=("Date", "count"),
    First_Date=("Date", "min"),
    Last_Date=("Date", "max"),
    Min_Close=("Close", "min"),
    Max_Close=("Close", "max"),
    Mean_Close=("Close", "mean"),
    Mean_Volume=("Volume", "mean"),
    Zero_Volume=("Volume", lambda x: (x == 0).sum())
)

display(stock_summary)

,Records,First_Date,Last_Date,Min_Close,Max_Close,Mean_Close,Mean_Volume,Zero_Volume
Ticker,,,,,,,,
ABX.TO,6744,2000-01-04,2026-11-06,7.900000,71.860001,29.453722,3.407982e+06,0
BMO.TO,6744,2000-01-04,2026-11-06,21.500000,257.880005,81.384961,1.836382e+06,0
BN.TO,6744,2000-01-04,2026-11-06,1.092395,67.900002,16.857449,3.679262e+06,0
CNQ.TO,6744,2000-01-04,2026-11-06,1.824192,78.419998,19.730614,9.111521e+06,1
CNR.TO,6744,2000-01-04,2026-11-06,5.608333,183.919998,71.285734,1.907072e+06,1
ENB.TO,6744,2000-01-04,2026-11-06,6.025000,80.209999,36.031077,3.449615e+06,1
RY.TO,6744,2000-01-04,2026-11-06,13.875000,305.739990,80.475226,3.236112e+06,0
SHOP.TO,2878,2015-05-21,2026-11-06,2.681000,252.529999,80.835101,2.560392e+06,0
SU.TO,6744,2000-01-04,2026-11-06,6.875000,111.419998,35.754091,4.833429e+06,1


In [13]:
# Calculate daily returns for each stock
stocks = stocks.sort_values(["Ticker", "Date"]).copy()

stocks["Daily_Return"] = (
    stocks.groupby("Ticker")["Adj Close"]
    .pct_change(fill_method=None) * 100
)

In [14]:
returns_summary = stocks.groupby("Ticker")["Daily_Return"].agg(
    Count="count",
    Mean="mean",
    Std="std",
    Min="min",
    Median="median",
    Max="max"
)

display(returns_summary.round(3))

,Count,Mean,Std,Min,Median,Max
Ticker,,,,,,
ABX.TO,6743,0.049,2.525,-16.394,0.000,32.463
BMO.TO,6743,0.061,1.397,-16.413,0.072,16.975
BN.TO,6743,0.079,1.751,-13.976,0.066,16.788
CNQ.TO,6743,0.090,2.391,-29.180,0.086,22.612
CNR.TO,6743,0.068,1.496,-11.116,0.083,10.544
ENB.TO,6743,0.060,1.382,-16.504,0.067,20.686
RY.TO,6743,0.066,1.329,-13.450,0.079,14.896
SHOP.TO,2877,0.213,3.649,-19.628,0.207,24.048
SU.TO,6743,0.072,2.254,-20.656,0.060,24.673


In [15]:
extreme_returns = stocks[
    stocks["Daily_Return"].abs() > 20
][
    ["Date", "Ticker", "Adj Close", "Daily_Return", "Volume"]
].sort_values("Daily_Return")

display(extreme_returns)

Price,Date,Ticker,Adj Close,Daily_Return,Volume
25302,2020-03-09,CNQ.TO,7.263219,-29.180444,28488146
25307,2020-03-16,CNQ.TO,4.894116,-27.254404,37304481
22469,2008-11-20,CNQ.TO,4.744168,-21.510872,30388636
55160,2020-03-12,SU.TO,15.789054,-20.656361,17858100
38795,2020-03-13,ENB.TO,28.268070,20.685822,14444400
49330,2023-11-02,SHOP.TO,82.080002,21.312441,6460300
49588,2024-11-12,SHOP.TO,152.259995,21.448505,6324600
49771,2025-08-06,SHOP.TO,212.820007,21.500344,4168100
25318,2020-03-31,CNQ.TO,6.786565,22.455468,45821459
25471,2020-11-09,CNQ.TO,9.546506,22.612363,33204962


In [16]:
print(
    stocks.groupby("Ticker")["Daily_Return"]
    .apply(lambda x: (x.abs() > 15).sum())
)

Ticker
ABX.TO      5
BMO.TO      2
BN.TO       2
CNQ.TO     13
CNR.TO      0
ENB.TO      3
RY.TO       0
SHOP.TO    17
SU.TO      11
TD.TO       1
Name: Daily_Return, dtype: int64


In [17]:
stocks["Adjustment_Ratio"] = (
    stocks["Adj Close"] / stocks["Close"]
)

display(
    stocks.groupby("Ticker")["Adjustment_Ratio"]
    .describe()
)

,count,mean,std,min,25%,50%,75%,max
Ticker,,,,,,,,
ABX.TO,6744.0,0.794809,0.088862,0.699586,0.712374,0.788642,0.841998,1.0
BMO.TO,6744.0,0.595138,0.196494,0.329708,0.408318,0.571795,0.749009,1.0
BN.TO,6744.0,0.843861,0.108268,0.614493,0.749976,0.859289,0.940780,1.0
CNQ.TO,6744.0,0.639910,0.137085,0.516681,0.541234,0.571837,0.692090,1.0
CNR.TO,6744.0,0.788679,0.104231,0.632985,0.694964,0.779493,0.872136,1.0
ENB.TO,6744.0,0.539162,0.188164,0.309550,0.390441,0.484156,0.652993,1.0
RY.TO,6744.0,0.629642,0.184683,0.377374,0.459470,0.600015,0.779150,1.0
SHOP.TO,2878.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.0
SU.TO,6744.0,0.680387,0.129554,0.555305,0.578444,0.617836,0.758615,1.0


In [18]:
print("Dividend events:")
display(
    stocks[stocks["Dividends"] > 0][
        ["Date", "Ticker", "Close", "Adj Close", "Dividends"]
    ].head(10)
)

print("Stock split events:")
display(
    stocks[stocks["Stock Splits"] > 0][
        ["Date", "Ticker", "Close", "Adj Close", "Stock Splits"]
    ]
)

Dividend events:


Price,Date,Ticker,Close,Adj Close,Dividends
983,2003-11-27,ABX.TO,28.900000,20.268818,0.073
1108,2004-05-27,ABX.TO,28.500000,20.041321,0.075
1235,2004-11-26,ABX.TO,29.990000,21.135954,0.065
1360,2005-05-27,ABX.TO,28.700001,20.289318,0.087
1487,2005-11-28,ABX.TO,31.809999,22.579216,0.129
1612,2006-05-29,ABX.TO,34.169998,24.341816,0.123
1739,2006-11-28,ABX.TO,34.060001,24.352427,0.123
1864,2007-05-29,ABX.TO,30.620001,22.010752,0.168
1991,2007-11-28,ABX.TO,41.419998,29.896496,0.168
2115,2008-05-28,ABX.TO,40.340000,29.263725,0.202


Stock split events:


Price,Date,Ticker,Close,Adj Close,Stock Splits
7037,2001-03-01,BMO.TO,37.700001,12.951309,2.000
14591,2004-05-19,BN.TO,3.529671,2.564955,1.500
15071,2006-04-17,BN.TO,7.181979,5.366194,1.500
15347,2007-05-22,BN.TO,10.916732,8.268441,1.500
16827,2013-04-15,BN.TO,8.831402,7.558238,1.033
17349,2015-05-13,BN.TO,15.564538,13.731295,1.500
18576,2020-04-02,BN.TO,22.225815,20.969034,1.500
19252,2022-12-12,BN.TO,30.726667,29.924658,1.237
19962,2025-10-10,BN.TO,60.290001,59.898788,1.500
21335,2004-05-19,CNQ.TO,4.689030,2.497998,2.000


In [19]:
stocks["Daily_Return"] = (
    stocks.groupby("Ticker")["Adj Close"]
    .pct_change() * 100
)

In [20]:
split_dates = stocks[stocks["Stock Splits"] > 0][
    ["Date", "Ticker", "Stock Splits"]
]

for _, row in split_dates.iterrows():

    ticker = row["Ticker"]
    date = row["Date"]

    event_data = stocks[
        (stocks["Ticker"] == ticker) &
        (stocks["Date"].between(
            date - pd.Timedelta(days=3),
            date + pd.Timedelta(days=3)
        ))
    ][["Date", "Ticker", "Close", "Adj Close", "Daily_Return"]]

    print(f"\n{ticker} | {date.date()} | Split: {row['Stock Splits']}")
    display(event_data)


BMO.TO | 2001-03-01 | Split: 2.0


Price,Date,Ticker,Close,Adj Close,Daily_Return
7034,2001-02-26,BMO.TO,40.974998,14.076390,2.823118
7035,2001-02-27,BMO.TO,39.294998,13.499245,-4.100096
7036,2001-02-28,BMO.TO,38.250000,13.140251,-2.659360
7037,2001-03-01,BMO.TO,37.700001,12.951309,-1.437887
7038,2001-03-02,BMO.TO,38.049999,13.071544,0.928358



BN.TO | 2004-05-19 | Split: 1.5


Price,Date,Ticker,Close,Adj Close,Daily_Return
14589,2004-05-17,BN.TO,3.397416,2.468848,0.651355
14590,2004-05-18,BN.TO,3.428333,2.491314,0.910006
14591,2004-05-19,BN.TO,3.529671,2.564955,2.955874
14592,2004-05-20,BN.TO,3.534824,2.568698,0.145963
14593,2004-05-21,BN.TO,3.612116,2.624866,2.186608



BN.TO | 2006-04-17 | Split: 1.5


Price,Date,Ticker,Close,Adj Close,Daily_Return
15071,2006-04-17,BN.TO,7.181979,5.366194,0.985308
15072,2006-04-18,BN.TO,7.249996,5.417017,0.947100
15073,2006-04-19,BN.TO,7.185071,5.368506,-0.895539
15074,2006-04-20,BN.TO,6.959378,5.199873,-3.141144


... (42 outputs left out)


SU.TO | 2008-05-12 | Split: 2.0


Price,Date,Ticker,Close,Adj Close,Daily_Return
52190,2008-05-09,SU.TO,62.889999,36.598648,-0.372244
52191,2008-05-12,SU.TO,63.980000,37.232952,1.733135
52192,2008-05-13,SU.TO,63.770000,37.110744,-0.328224
52193,2008-05-14,SU.TO,63.369999,36.877964,-0.627259
52194,2008-05-15,SU.TO,64.879997,37.756710,2.382849



TD.TO | 2014-02-03 | Split: 2.0


Price,Date,Ticker,Close,Adj Close,Daily_Return
60371,2014-01-31,TD.TO,48.160000,28.893913,-0.874770
60372,2014-02-03,TD.TO,47.889999,28.731947,-0.560555
60373,2014-02-04,TD.TO,47.619999,28.569939,-0.563861
60374,2014-02-05,TD.TO,47.860001,28.713936,0.504016
60375,2014-02-06,TD.TO,48.630001,29.175907,1.608875
